In [17]:
from IPython import get_ipython
import pandas as pd 
import os
from pathlib import Path
from pandas import DataFrame

try:
    from src.data.utils import gerar_df_dic,carregar_parquet_local,salvar_parquet_local
except ModuleNotFoundError:
    # Se falhar, instala o pacote local e importa
    get_ipython().run_line_magic('pip', 'install -e ..')
    from src.data.utils import gerar_df_dic,carregar_parquet_local,salvar_parquet_local

In [3]:
#Nao ocultar as informações de descrição 
pd.set_option('display.max_colwidth', None)

ano = 2023

#dados
raw_alunos = carregar_parquet_local(ano,'TS_ALUNO')
raw_itens =  carregar_parquet_local(ano,'TS_ITEM') 
raw_uf  = carregar_parquet_local(ano,'TS_ESTADO') 
raw_municipio  = carregar_parquet_local(ano,'TS_MUNICIPIO') 


#Dicionarios
dicionario_alunos =   carregar_parquet_local(ano,'TS_ALUNO', ler_dicionario=True)
dicionario_itens =   carregar_parquet_local(ano,'TS_ITEM', ler_dicionario=True)
dicionario_municipio =   carregar_parquet_local(ano,'TS_ESTADO', ler_dicionario=True)
dicionario_uf =   carregar_parquet_local(ano,'TS_MUNICIPIO', ler_dicionario=True)

Lendo Parquet em: ..\data\bronze\ano=2023\dados\TS_ALUNO.parquet
Lendo Parquet em: ..\data\bronze\ano=2023\dados\TS_ITEM.parquet
Lendo Parquet em: ..\data\bronze\ano=2023\dados\TS_ESTADO.parquet
Lendo Parquet em: ..\data\bronze\ano=2023\dados\TS_MUNICIPIO.parquet
Lendo Parquet em: ..\data\bronze\ano=2023\dicionario\dicionario_TS_ALUNO.parquet
Lendo Parquet em: ..\data\bronze\ano=2023\dicionario\dicionario_TS_ITEM.parquet
Lendo Parquet em: ..\data\bronze\ano=2023\dicionario\dicionario_TS_ESTADO.parquet
Lendo Parquet em: ..\data\bronze\ano=2023\dicionario\dicionario_TS_MUNICIPIO.parquet


Precisamos fazer o Merge levando em consideração o ID_TIPO_REDE, para integrar a tabela dos alunos, primeiro faremos um pivot para a tabela de UF e Municipio 

In [24]:
def processar_camada_silver(ano: int | str) -> pd.DataFrame:
    """
    Processa e integra as bases da camada Bronze, gerando o DataFrame final
    da camada Silver enriquecido com dados de município e UF para o ano informado.
    """
    # 1. Carrega automaticamente as bases Bronze locais
    print(f"\n[INFO] Carregando dados do ano {ano}...")
    raw_alunos = carregar_parquet_local(ano, 'TS_ALUNO')
    raw_municipio = carregar_parquet_local(ano, 'TS_MUNICIPIO')
    raw_uf = carregar_parquet_local(ano, 'TS_ESTADO')
    
    MAPA_REDES = {
        0: 'TOTAL',
        1: 'FEDERAL',
        2: 'ESTADUAL',
        3: 'MUNICIPAL',
        4: 'PRIVADA',
        5: 'PUBLICA_EST_MUN',
        6: 'PUBLICA_TOTAL'
    }
    
    METRICAS = ['VL_MEDIA_LP', 'PC_ALUNO_ALFABETIZADO']
    
    # === A. PREPARAÇÃO DO MUNICÍPIO =========================================
    print(f"[INFO] Processando dimensões do Município para {ano}...")
    CHAVES_MUN = ['NU_ANO_AVALIACAO', 'CO_UF', 'CO_MUNICIPIO', 'TP_SERIE']
    REDES_MUN = list(MAPA_REDES.values())
    
    municipio_prep = (
        raw_municipio[CHAVES_MUN + ['ID_TIPO_REDE'] + METRICAS]
        .assign(REDE_NOME=lambda df: df['ID_TIPO_REDE'].map(MAPA_REDES))
        .query('REDE_NOME.notna()')   
        .drop(columns='ID_TIPO_REDE')
    )
    
    municipio_pivot = municipio_prep.pivot(index=CHAVES_MUN, columns='REDE_NOME', values=METRICAS)
    municipio_pivot.columns = [f"{metrica}_{rede}_MUNICIPIO" for metrica, rede in municipio_pivot.columns]
    municipio_dim = municipio_pivot.reset_index()
    
    # Garante consistência de colunas (trata anos sem ID 0)
    colunas_esperadas_mun = [f"{metrica}_{rede}_MUNICIPIO" for metrica in METRICAS for rede in REDES_MUN]
    municipio_dim = municipio_dim.reindex(columns=CHAVES_MUN + colunas_esperadas_mun)
    
    # Fallback TOTAL -> PUBLICA_EST_MUN
    for metrica in METRICAS:
        col_total = f"{metrica}_TOTAL_MUNICIPIO"
        col_publica = f"{metrica}_PUBLICA_EST_MUN_MUNICIPIO"
        municipio_dim[col_total] = municipio_dim[col_total].fillna(municipio_dim[col_publica])

    # === B. PREPARAÇÃO DA UF ===============================================
    print(f"[INFO] Processando dimensões da UF para {ano}...")
    CHAVES_UF = ['NU_ANO_AVALIACAO', 'CO_UF', 'TP_SERIE']
    REDES_UF = list(MAPA_REDES.values())
    
    uf_prep = (
        raw_uf[CHAVES_UF + ['ID_TIPO_REDE'] + METRICAS]
        .assign(REDE_NOME=lambda df: df['ID_TIPO_REDE'].map(MAPA_REDES))
        .query('REDE_NOME.notna()')
        .drop(columns='ID_TIPO_REDE')
    )
    
    uf_pivot = uf_prep.pivot(index=CHAVES_UF, columns='REDE_NOME', values=METRICAS)
    uf_pivot.columns = [f"{metrica}_{rede}_UF" for metrica, rede in uf_pivot.columns]
    uf_dim = uf_pivot.reset_index()
    
    # Garante consistência de colunas da UF
    colunas_esperadas_uf = [f"{metrica}_{rede}_UF" for metrica in METRICAS for rede in REDES_UF]
    uf_dim = uf_dim.reindex(columns=CHAVES_UF + colunas_esperadas_uf)
    
    # Fallback TOTAL -> PUBLICA_EST_MUN
    for metrica in METRICAS:
        col_total = f"{metrica}_TOTAL_UF"
        col_publica = f"{metrica}_PUBLICA_EST_MUN_UF"
        uf_dim[col_total] = uf_dim[col_total].fillna(uf_dim[col_publica])

    # === C. JUNCÕES (MERGE) =================================================
    print(f"[INFO] Mesclando dados na tabela de alunos...")
    df_silver = pd.merge(
        raw_alunos,
        municipio_dim,
        on=['NU_ANO_AVALIACAO', 'CO_UF', 'CO_MUNICIPIO', 'TP_SERIE'],
        how='left'
    )
    
    df_silver = pd.merge(
        df_silver,
        uf_dim,
        on=['NU_ANO_AVALIACAO', 'CO_UF', 'TP_SERIE'],
        how='left'
    )

    #  CRIAÇÃO DE COLUNAS DERIVADAS E LIMPEZA 
    print(f"[INFO] Calculando desvios e limpando colunas vazias...")
    df_silver['DESVIO_MEDIA_MUNICIPIO'] = df_silver['VL_PROFICIENCIA_LP'] - df_silver['VL_MEDIA_LP_TOTAL_MUNICIPIO']
    df_silver['DESVIO_MEDIA_UF'] = df_silver['VL_PROFICIENCIA_LP'] - df_silver['VL_MEDIA_LP_TOTAL_UF']
    
    # Dropa as colunas de redes nulas
    colunas_para_remover = [
        'VL_MEDIA_LP_FEDERAL_MUNICIPIO', 'PC_ALUNO_ALFABETIZADO_FEDERAL_MUNICIPIO',
        'VL_MEDIA_LP_PRIVADA_MUNICIPIO', 'PC_ALUNO_ALFABETIZADO_PRIVADA_MUNICIPIO',
        'VL_MEDIA_LP_PUBLICA_TOTAL_MUNICIPIO', 'PC_ALUNO_ALFABETIZADO_PUBLICA_TOTAL_MUNICIPIO',
        'VL_MEDIA_LP_FEDERAL_UF', 'PC_ALUNO_ALFABETIZADO_FEDERAL_UF',
        'VL_MEDIA_LP_PRIVADA_UF', 'PC_ALUNO_ALFABETIZADO_PRIVADA_UF',
        'VL_MEDIA_LP_PUBLICA_TOTAL_UF', 'PC_ALUNO_ALFABETIZADO_PUBLICA_TOTAL_UF'
    ]
    df_silver = df_silver.drop(columns=colunas_para_remover, errors='ignore')

    print(f"[INFO] Ano {ano} processado com sucesso!")
    return df_silver


In [ ]:
# Lista de anos do projeto
anos_projeto = [2023, 2024, 2025]

for ano in anos_projeto:
    df_silver_ano = processar_camada_silver(ano)
    caminho_prata = Path(f"../data/prata/ano={ano}/alunos_prata.parquet")
    
    print(f"Salvando arquivo em: {caminho_prata}...")
    salvar_parquet_local(df_silver_ano, caminho_prata, index=False)

print("\n>>> Pipeline Silver concluído com sucesso para todos os anos! <<<")



[INFO] Carregando dados do ano 2023...
Lendo Parquet em: ..\data\bronze\ano=2023\dados\TS_ALUNO.parquet
Lendo Parquet em: ..\data\bronze\ano=2023\dados\TS_MUNICIPIO.parquet
Lendo Parquet em: ..\data\bronze\ano=2023\dados\TS_ESTADO.parquet
[INFO] Processando dimensões do Município para 2023...
[INFO] Processando dimensões da UF para 2023...
[INFO] Mesclando dados na tabela de alunos...
[INFO] Calculando desvios e limpando colunas vazias...
[INFO] Ano 2023 processado com sucesso!
[INFO] Salvando arquivo em: ..\data\prata\ano=2023\alunos_prata.parquet...
Arquivo salvo localmente em: ..\data\prata\ano=2023\alunos_prata.parquet

[INFO] Carregando dados do ano 2024...
Lendo Parquet em: ..\data\bronze\ano=2024\dados\TS_ALUNO.parquet
Lendo Parquet em: ..\data\bronze\ano=2024\dados\TS_MUNICIPIO.parquet
Lendo Parquet em: ..\data\bronze\ano=2024\dados\TS_ESTADO.parquet
[INFO] Processando dimensões do Município para 2024...
[INFO] Processando dimensões da UF para 2024...
[INFO] Mesclando dados na

In [21]:
df_silver_ano = processar_camada_silver(ano)


[INFO] Carregando dados do ano 2025...
Lendo Parquet em: ..\data\bronze\ano=2025\dados\TS_ALUNO.parquet
Lendo Parquet em: ..\data\bronze\ano=2025\dados\TS_MUNICIPIO.parquet
Lendo Parquet em: ..\data\bronze\ano=2025\dados\TS_ESTADO.parquet
[INFO] Processando dimensões do Município para 2025...
[INFO] Processando dimensões da UF para 2025...
[INFO] Mesclando dados na tabela de alunos...
[INFO] Calculando desvios e limpando colunas vazias...
[INFO] Ano 2025 processado com sucesso!


In [23]:
df_silver_ano.columns

Index(['NU_ANO_AVALIACAO', 'CO_UF', 'SG_UF', 'ID_ALUNO', 'TP_SERIE',
       'ID_ESCOLA', 'TP_DEPENDENCIA', 'CO_MUNICIPIO', 'NO_MUNICIPIO',
       'IN_PRESENCA_LP', 'IN_PREENCHIMENTO_LP', 'CO_CADERNO_LP', 'CO_BLOCO_1',
       'TX_RESPOSTA_BLOCO_1', 'TX_GABARITO_BLOCO_1', 'CO_BLOCO_2',
       'TX_RESPOSTA_BLOCO_2', 'TX_GABARITO_BLOCO_2', 'CO_BLOCO_3',
       'TX_RESPOSTA_BLOCO_3', 'TX_GABARITO_BLOCO_3', 'CO_BLOCO_4',
       'TX_RESPOSTA_BLOCO_4', 'TX_GABARITO_BLOCO_4', 'VL_PESO_ALUNO_LP',
       'VL_PROFICIENCIA_LP', 'IN_ALFABETIZADO', 'VL_MEDIA_LP_TOTAL_MUNICIPIO',
       'VL_MEDIA_LP_ESTADUAL_MUNICIPIO', 'VL_MEDIA_LP_MUNICIPAL_MUNICIPIO',
       'VL_MEDIA_LP_PUBLICA_EST_MUN_MUNICIPIO',
       'PC_ALUNO_ALFABETIZADO_TOTAL_MUNICIPIO',
       'PC_ALUNO_ALFABETIZADO_ESTADUAL_MUNICIPIO',
       'PC_ALUNO_ALFABETIZADO_MUNICIPAL_MUNICIPIO',
       'PC_ALUNO_ALFABETIZADO_PUBLICA_EST_MUN_MUNICIPIO',
       'VL_MEDIA_LP_TOTAL_UF', 'VL_MEDIA_LP_ESTADUAL_UF',
       'VL_MEDIA_LP_MUNICIPAL_UF', 'VL

O Pipeline parte por parte para debug posterior 

In [ ]:
ano = 2024

In [ ]:
#  Pivot da tabela munipicio

MAPA_REDES = {
    0: 'TOTAL',
    1: 'FEDERAL',
    2: 'ESTADUAL',
    3: 'MUNICIPAL',
    4: 'PRIVADA',
    5: 'PUBLICA_EST_MUN',
    6: 'PUBLICA_TOTAL'
}


METRICAS = ['VL_MEDIA_LP', 'PC_ALUNO_ALFABETIZADO']
CHAVES    = ['NU_ANO_AVALIACAO', 'CO_UF', 'CO_MUNICIPIO', 'TP_SERIE']
REDES     = list(MAPA_REDES.values())  # ordem esperada de colunas

# Filtra, mapeia as redes
municipio_prep = (
    raw_municipio[CHAVES + ['ID_TIPO_REDE'] + METRICAS]
    .assign(REDE_NOME=lambda df: df['ID_TIPO_REDE'].map(MAPA_REDES))
    .query('REDE_NOME.notna()')   
    .drop(columns='ID_TIPO_REDE')
)

# Pivot: uma coluna por (métrica × rede) 
municipio_pivot = municipio_prep.pivot(
    index=CHAVES,
    columns='REDE_NOME',
    values=METRICAS
)

# Achata MultiIndex → "VL_MEDIA_LP_ESTADUAL_MUNICIPIO" etc. ──────────
municipio_pivot.columns = [
    f"{metrica}_{rede}_MUNICIPIO"
    for metrica, rede in municipio_pivot.columns
]
municipio_dim = municipio_pivot.reset_index()

# Garante todas as colunas esperadas (resiliente a anos incompletos) ──
colunas_esperadas = [
    f"{metrica}_{rede}_MUNICIPIO"
    for metrica in METRICAS
    for rede in REDES
]
municipio_dim = municipio_dim.reindex(
    columns=CHAVES + colunas_esperadas  # ordem determinística
)

# ── 5. Fallback: TOTAL ausente → usa PUBLICA_EST_MUN ─────────────────────
for metrica in METRICAS:
    col_total   = f"{metrica}_TOTAL_MUNICIPIO"
    col_publica = f"{metrica}_PUBLICA_EST_MUN_MUNICIPIO"
    municipio_dim[col_total] = municipio_dim[col_total].fillna(
        municipio_dim[col_publica]
    )

In [ ]:
#Pivot da tabela UF

uf_prep = raw_uf[
    ['NU_ANO_AVALIACAO', 'CO_UF', 'TP_SERIE', 'ID_TIPO_REDE', 'VL_MEDIA_LP', 'PC_ALUNO_ALFABETIZADO']
].copy()

#utilizar o dicionario para converter os IDs em  texto
uf_prep['REDE_NOME'] = uf_prep['ID_TIPO_REDE'].map(MAPA_REDES)

# Criar um pivot da tabela 
uf_pivot = uf_prep.pivot(
    index=['NU_ANO_AVALIACAO', 'CO_UF', 'TP_SERIE'],
    columns='REDE_NOME',
    values=['VL_MEDIA_LP', 'PC_ALUNO_ALFABETIZADO']
)

# Gerar MultiIndex das colunas
uf_pivot.columns = [f"{metrica}_{rede}_UF" for metrica, rede in uf_pivot.columns]
uf_dim = uf_pivot.reset_index()



uf_dim.head(3)


,NU_ANO_AVALIACAO,CO_UF,TP_SERIE,VL_MEDIA_LP_ESTADUAL_UF,VL_MEDIA_LP_MUNICIPAL_UF,VL_MEDIA_LP_PUBLICA_EST_MUN_UF,PC_ALUNO_ALFABETIZADO_ESTADUAL_UF,PC_ALUNO_ALFABETIZADO_MUNICIPAL_UF,PC_ALUNO_ALFABETIZADO_PUBLICA_EST_MUN_UF
0,2023,11,2,751.4731,760.1971,759.4357,58.65,65.17,64.60
1,2023,13,2,746.2058,733.6637,736.4687,62.63,49.20,52.20
2,2023,15,2,741.3548,732.6697,733.2877,57.07,47.71,48.38


In [ ]:
# Merge dos alunos com munipio e UF
df_silver = pd.merge(
    raw_alunos,
    municipio_dim,
    on=['NU_ANO_AVALIACAO', 'CO_UF', 'CO_MUNICIPIO', 'TP_SERIE'],
    how='left'
)

# ── Etapa 4: + UF 
df_silver = pd.merge(
    df_silver,
    uf_dim,
    on=['NU_ANO_AVALIACAO', 'CO_UF', 'TP_SERIE'],
    how='left'
)



In [7]:
df_silver.head(5)

,NU_ANO_AVALIACAO,CO_UF,SG_UF,ID_ALUNO,TP_SERIE,ID_ESCOLA,TP_DEPENDENCIA,CO_MUNICIPIO,NO_MUNICIPIO,IN_PRESENCA_LP,...,PC_ALUNO_ALFABETIZADO_MUNICIPAL_MUNICIPIO,PC_ALUNO_ALFABETIZADO_PRIVADA_MUNICIPIO,PC_ALUNO_ALFABETIZADO_PUBLICA_EST_MUN_MUNICIPIO,PC_ALUNO_ALFABETIZADO_PUBLICA_TOTAL_MUNICIPIO,VL_MEDIA_LP_ESTADUAL_UF,VL_MEDIA_LP_MUNICIPAL_UF,VL_MEDIA_LP_PUBLICA_EST_MUN_UF,PC_ALUNO_ALFABETIZADO_ESTADUAL_UF,PC_ALUNO_ALFABETIZADO_MUNICIPAL_UF,PC_ALUNO_ALFABETIZADO_PUBLICA_EST_MUN_UF
0,2023,11,RO,11008701,2,60000001,3,1100205,Porto Velho,0,...,65.06,NaN,64.47,NaN,751.4731,760.1971,759.4357,58.65,65.17,64.6
1,2023,11,RO,11008695,2,60000001,3,1100205,Porto Velho,1,...,65.06,NaN,64.47,NaN,751.4731,760.1971,759.4357,58.65,65.17,64.6
2,2023,11,RO,11008687,2,60000001,3,1100205,Porto Velho,0,...,65.06,NaN,64.47,NaN,751.4731,760.1971,759.4357,58.65,65.17,64.6
3,2023,11,RO,11008682,2,60000001,3,1100205,Porto Velho,1,...,65.06,NaN,64.47,NaN,751.4731,760.1971,759.4357,58.65,65.17,64.6
4,2023,11,RO,11008729,2,60000001,3,1100205,Porto Velho,0,...,65.06,NaN,64.47,NaN,751.4731,760.1971,759.4357,58.65,65.17,64.6


In [8]:
df_silver.shape

(1747439, 35)

In [10]:
itens_na = pd.DataFrame(df_silver.isna().sum(), columns=['Qtd_Nulos'])
itens_na_filtrado = itens_na[itens_na['Qtd_Nulos'] > 0]

itens_na_filtrado.sort_values(by='Qtd_Nulos')



,Qtd_Nulos
PC_ALUNO_ALFABETIZADO_PUBLICA_EST_MUN_MUNICIPIO,446
VL_MEDIA_LP_TOTAL_MUNICIPIO,446
PC_ALUNO_ALFABETIZADO_TOTAL_MUNICIPIO,446
VL_MEDIA_LP_PUBLICA_EST_MUN_MUNICIPIO,446
PC_ALUNO_ALFABETIZADO_MUNICIPAL_MUNICIPIO,2413
VL_MEDIA_LP_MUNICIPAL_MUNICIPIO,2413
PC_ALUNO_ALFABETIZADO_ESTADUAL_UF,180056
VL_MEDIA_LP_ESTADUAL_UF,180056
VL_PESO_ALUNO_LP,244630
VL_PROFICIENCIA_LP,244630


Temos um grande numero de Nulos na base, isso se da porque algumas escolas nao participaram da prova, nao temos nada na esfera Federal (praticamente nao existem escolas federais de ensino fundamental no Brasil) 
as escolas privadas também nao participam da prova

**VL_MEDIA_LP_TOTAL_MUNICIPIO e PC_ALFAB_MUNICIPIO:** <br><br>
O
O que significa: Representa alunos que estudam em municípios extremamente pequenos cujos dados de médias foram ocultados pelo INEP por segredo estatístico (quando a cidade tem pouquíssimos alunos, o INEP não divulga as médias para não expor notas individuais). São perdas aceitáveis e muito pequenas (apenas 446 alunos no Brasil inteiro).

**VL_MEDIA_LP_MUNICIPAL_MUNICIPIO:**<br><br>
O que significa: Alunos que estudam em cidades onde não existem escolas municipais oferecendo o 2º ano do Ensino Fundamental. Nesses locais, toda a educação de alfabetização é concentrada na Rede Estadual.


**VL_PROFICIENCIA_LP e VL_PESO_ALUNO_LP**<br><br>
O que significa: Como analisamos anteriormente, são os alunos que faltaram no dia da prova ou tiveram suas avaliações anuladas/desconsideradas pelo INEP. Eles não possuem nota individual de proficiência.



**VL_MEDIA_LP_ESTADUAL_UF:** <br><br>
O que significa:
Ocorre principalmente no Distrito Federal (DF), onde não existe a divisão entre rede Estadual e Municipal (tudo é rede Distrital). Por conta disso, a categoria "Estadual" do DF fica inteiramente nula.
Estados onde a alfabetização foi 100% municipalizada e a secretaria estadual não gerencia nenhuma escola de 2º ano.


**VL_MEDIA_LP_ESTADUAL_MUNICIPIO**<br><br>
O que significa: Reflete o processo brasileiro de municipalização da alfabetização. Na imensa maioria das cidades do interior do país, o estado repassou a responsabilidade dos anos iniciais do ensino fundamental (1º ao 5º ano) inteiramente para as prefeituras. Como não existem escolas estaduais de 2º ano nesses municípios, essa coluna fica vazia para mais de 1 milhão de registros.



**VL_MEDIA_LP_FEDERAL_MUNICIPIO e VL_MEDIA_LP_PRIVADA_MUNICIPIO:** <br><br>
Ausência de Dados Federais e Privados  <br>
O que significa: Praticamente toda a base de alunos está nula nestas colunas porque a rede Federal e Privada não participam ou não possuem turmas de 2º ano do Ensino Fundamental registradas na avaliação na quase totalidade das cidades do Brasil.



In [11]:
# Remove as colunas que estão 
colunas_para_remover = [
    'VL_MEDIA_LP_FEDERAL_MUNICIPIO',
    'PC_ALUNO_ALFABETIZADO_FEDERAL_MUNICIPIO',
    'VL_MEDIA_LP_PRIVADA_MUNICIPIO',
    'PC_ALUNO_ALFABETIZADO_PRIVADA_MUNICIPIO',
    'VL_MEDIA_LP_PUBLICA_TOTAL_MUNICIPIO',
    'PC_ALUNO_ALFABETIZADO_PUBLICA_TOTAL_MUNICIPIO'
]

df_silver = df_silver.drop(columns=colunas_para_remover, errors='ignore')


In [12]:
df_silver['ANO_AVALIACAO'] = ano

In [13]:
df_silver.head(5).T

,0,1,2,3,4
NU_ANO_AVALIACAO,2023,2023,2023,2023,2023
CO_UF,11,11,11,11,11
SG_UF,RO,RO,RO,RO,RO
ID_ALUNO,11008701,11008695,11008687,11008682,11008729
TP_SERIE,2,2,2,2,2
ID_ESCOLA,60000001,60000001,60000001,60000001,60000001
TP_DEPENDENCIA,3,3,3,3,3
CO_MUNICIPIO,1100205,1100205,1100205,1100205,1100205
NO_MUNICIPIO,Porto Velho,Porto Velho,Porto Velho,Porto Velho,Porto Velho
IN_PRESENCA_LP,0,1,0,1,0


In [14]:
itens_na = pd.DataFrame(df_silver.isna().sum(), columns=['Qtd_Nulos'])
itens_na_filtrado = itens_na[itens_na['Qtd_Nulos'] > 0]

itens_na_filtrado.sort_values(by='Qtd_Nulos')



,Qtd_Nulos
VL_MEDIA_LP_TOTAL_MUNICIPIO,446
VL_MEDIA_LP_PUBLICA_EST_MUN_MUNICIPIO,446
PC_ALUNO_ALFABETIZADO_TOTAL_MUNICIPIO,446
PC_ALUNO_ALFABETIZADO_PUBLICA_EST_MUN_MUNICIPIO,446
VL_MEDIA_LP_MUNICIPAL_MUNICIPIO,2413
PC_ALUNO_ALFABETIZADO_MUNICIPAL_MUNICIPIO,2413
VL_MEDIA_LP_ESTADUAL_UF,180056
PC_ALUNO_ALFABETIZADO_ESTADUAL_UF,180056
VL_PESO_ALUNO_LP,244630
VL_PROFICIENCIA_LP,244630


In [ ]:
from pathlib import Path
from src.data.utils import salvar_parquet_local

# 1. Define o caminho de destino na camada Prata
# Estrutura: ../data/prata/ano=2025/alunos_prata.parquet
caminho_prata = Path(f"../data/prata/ano={ano}/alunos_prata.parquet")

# 2. Salva o DataFrame no formato Parquet
# Usamos index=False para não salvar o índice sequencial padrão do Pandas no arquivo
salvar_parquet_local(df_silver, caminho_prata, index=False)
